In [1]:
import os
import json
import numpy as np
from openai import OpenAI
from google.colab import userdata
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

In [3]:
OPENAI_EMBEDDING_MODEL='text-embedding-3-small'
# embedding_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
embedding_model = OPENAI_EMBEDDING_MODEL
embedding_dim = 1536  # Number of dimensions in the embedding vector
print(f"Using OpenAI model: {embedding_model}")
print(f"Embedding dimension: {embedding_dim}")


Using OpenAI model: text-embedding-3-small
Embedding dimension: 1536


In [5]:
# =============================================================================
# LOAD DATA: Support Tickets
# =============================================================================
print("\nLoading support tickets...")
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"Loaded {len(tickets)} support tickets")


Loading support tickets...
Loaded 20 support tickets


In [6]:
# Display sample ticket - understand what we're working with
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")


SAMPLE TICKET:
ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment....
Category: Authentication
Priority: High


In [7]:
print("\n" + "="*80)
print("PART 1: Generating Embeddings")
print("="*80)

ticket_texts = [
    f"{ticket['title']}. {ticket['description']}"
    for ticket in tickets
]
print(f"\nticket_texts: '{str(ticket_texts)}'")


print("\nGenerating embeddings for all tickets...")
response = client.embeddings.create(input=ticket_texts, model=embedding_model)

# Convert API response to NumPy array for mathematical operations
# Shape: (num_tickets, 1536) - each row is one ticket's embedding
embeddings = np.array([data.embedding for data in response.data])
print(f"✓ Generated embeddings with shape: {embeddings.shape}")
print(f"  ({len(tickets)} tickets × {embedding_dim} dimensions)")


PART 1: Generating Embeddings

ticket_texts: '["Users unable to log in after password reset. Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.", "Database connection timeout in production. Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.", "Payment processing fails for international cards. Customers using non-US credit cards receiving 'Payment declined' errors. Payment gateway returns error code 402. Issue only affects cards issued outside United States.", 'Mobile app crashes on iOS 17. App crashes immediately on launch for users who upgraded to iOS 17. Crash reports show EXC_BAD_ACCESS in networking module. Affects approximately 30% of iOS user base.', 'Memory leak in background worker process. Background job processor showing ste

In [8]:
print(f"\nFirst 10 values of embedding for ticket 1:")
print(embeddings[0][:10])
print("  (These 1536 numbers encode the semantic meaning of the text)")


First 10 values of embedding for ticket 1:
[-0.00806427  0.05163574  0.05493164  0.02612305 -0.01281738 -0.02459717
 -0.0045929   0.06225586 -0.00149059  0.00540543]
  (These 1536 numbers encode the semantic meaning of the text)


In [9]:
print("\n" + "="*80)
print("PART 2: Computing Similarity Scores")
print("="*80)

# Create a search query - this is what a user might type
query = "Database is running very slowly"
print(f"\nSearch Query: '{query}'")

# -----------------------------------------------------------------------------
# Generate embedding for the query
# -----------------------------------------------------------------------------

query_response = client.embeddings.create(input=[query], model=embedding_model)
query_embedding = np.array([query_response.data[0].embedding])
print(f"Query embedding shape: {query_embedding.shape}")  # (1,


PART 2: Computing Similarity Scores

Search Query: 'Database is running very slowly'
Query embedding shape: (1, 1536)


In [10]:
# -----------------------------------------------------------------------------
# Compute cosine similarity between query and ALL tickets
# -----------------------------------------------------------------------------

similarities = cosine_similarity(query_embedding, embeddings)[0]
print(f"\nComputed similarities look like {similarities}")
print(f"\nComputed similarity scores for {len(similarities)} tickets")
print(f"Similarity range: [{similarities.min():.4f}, {similarities.max():.4f}]")


Computed similarities look like [0.16099904 0.45114916 0.1698147  0.17008409 0.27186581 0.15014405
 0.28954392 0.29553778 0.31182958 0.52453241 0.20436003 0.27680352
 0.23007134 0.27264147 0.25019505 0.15467512 0.2713283  0.27453902
 0.26371133 0.29924803]

Computed similarity scores for 20 tickets
Similarity range: [0.1501, 0.5245]


In [12]:
# ============================================================================
# PART 3: Retrieve Most Similar Tickets
# ============================================================================

print("\n" + "="*80)
print("PART 3: Finding Most Similar Tickets")
print("="*80)

# Get top-10 most similar tickets
top_k = 10

top_indices = np.argsort(similarities)[::-1][:top_k]

print(f"\nTop {top_k} most similar tickets to query: '{query}'")
print(f"\ntop_indices: '{top_indices}'")
print("-" * 80)

for rank, idx in enumerate(top_indices, 1):
    ticket = tickets[idx]
    score = similarities[idx]

    print(f"\n#{rank} - Similarity: {score:.4f}")
    print(f"Ticket ID: {ticket['ticket_id']}")
    print(f"Title: {ticket['title']}")
    print(f"Category: {ticket['category']} | Priority: {ticket['priority']}")
    print(f"Description: {ticket['description'][:150]}...")



PART 3: Finding Most Similar Tickets

Top 10 most similar tickets to query: 'Database is running very slowly'

top_indices: '[ 9  1  8 19  7  6 11 17 13  4]'
--------------------------------------------------------------------------------

#1 - Similarity: 0.5245
Ticket ID: TICK-010
Title: Dashboard loading extremely slowly
Category: Performance | Priority: High
Description: Admin dashboard takes 30+ seconds to load. Browser DevTools show multiple slow database queries. Page becomes unresponsive during load. Affects all us...

#2 - Similarity: 0.4511
Ticket ID: TICK-002
Title: Database connection timeout in production
Category: Database | Priority: Critical
Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20...

#3 - Similarity: 0.3118
Ticket ID: TICK-009
Title: API rate limiting too aggressive
Category: API | Priority: Medium
Description: Legitimate users hitting rate limit